# PH-SHOWOA · Python Full-GPU Tensorized SA-RCRS-GRASP Solver Benchmark

**Mục tiêu**: Chạy thuật toán PH-SHOWOA Python Full-GPU Tensorized (`src_python_gpu_SA_RCRS_GRASP`) tăng tốc PyTorch GPU (CUDA) với phương pháp khởi tạo kết hợp **SA-RCRS-GRASP** trên 15 bộ dữ liệu VRPSPDTW chuẩn của Wang & Chen.

**Thông số thử nghiệm**:
- `Compute Backend` = `cuda` (PyTorch 3D Tensorized CUDA GPU Evaluation & Acceleration)
- `Init` = `sa_rcrs_grasp` (Tự động kèm RCRS-GRASP RCL + 25 vòng SA Post-refinement)
- `Popsize` = 36 (6x6 Perfect Square Island Grid)
- `Max-iteration` = 1000
- `Runs` = 30

**GPU khuyên dùng trên Kaggle**: **NVIDIA Tesla T4** / **NVIDIA P100** / **NVIDIA L4**

## Cell 1 – Clone hoặc Cập nhật Repo từ GitHub

In [ ]:
import os
if os.path.exists("ph-showoa"):
    !cd ph-showoa && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/Welkie/ph-showoa.git


## Cell 2 – Kiểm tra GPU & Môi trường PyTorch CUDA

In [ ]:
!nvidia-smi
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU runtime is required: enable a CUDA accelerator before running the benchmark.")
torch.cuda.set_device(0)
print("Device name:", torch.cuda.get_device_name(0))
print("CUDA capability:", torch.cuda.get_device_capability(0))

## Cell 3 – Build và xác minh Native CUDA Full-GPU Solver

Notebook này bắt buộc build `src_python_gpu_SA_RCRS_GRASP/cpp_native` với `ENABLE_CUDA=ON`. Python chỉ điều phối process và nhận kết quả cuối; toàn bộ population, SA, RCRS-GRASP, objective, local search và migration chạy trong CUDA kernels.

In [ ]:
from pathlib import Path
import os
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required before building the native solver.")
if subprocess.run(["which", "nvcc"], capture_output=True, text=True).returncode != 0:
    raise RuntimeError("nvcc is unavailable. Enable a Kaggle GPU accelerator with CUDA Toolkit.")

repo = Path("ph-showoa" if Path("ph-showoa").exists() else ".").resolve()
native_dir = repo / "src_python_gpu_SA_RCRS_GRASP" / "cpp_native"
build_dir = native_dir / "build"
binary = build_dir / "phshowoa_cpp"

capability = torch.cuda.get_device_capability(0)
cuda_arch = f"{capability[0]}{capability[1]}"
env = os.environ.copy()
env["CMAKE_BUILD_PARALLEL_LEVEL"] = "2"

subprocess.run(
    [
        "cmake",
        "-S", str(native_dir),
        "-B", str(build_dir),
        "-DENABLE_CUDA=ON",
        "-DCMAKE_BUILD_TYPE=Release",
        f"-DCMAKE_CUDA_ARCHITECTURES={cuda_arch}",
    ],
    check=True,
    env=env,
)
subprocess.run(
    ["cmake", "--build", str(build_dir), "--config", "Release", "--parallel", "2"],
    check=True,
    env=env,
)

if not binary.exists():
    raise RuntimeError(f"Native CUDA binary was not produced: {binary}")

probe = subprocess.run([str(binary), "--check_cuda"], capture_output=True, text=True)
probe_output = (probe.stdout or "") + (probe.stderr or "")
print(probe_output)
if "CUDA_ENABLED" not in probe_output:
    raise RuntimeError("Native solver is not CUDA-enabled; benchmark was blocked.")
print(f"Native CUDA solver verified: {binary}")

## Cell 4 – Batch runner Python Full-GPU Tensorized SA-RCRS-GRASP (15 bộ dữ liệu)


In [ ]:
import glob
import os
import re
import subprocess
import sys
import time as T
from collections import deque

import pandas as pd

DATASETS = [
    "rcdp1001", "rcdp5001", "rcdp5007", "rcdp5004", "rcdp101",
    "cdp103", "rcdp205", "rdp210", "rcdp207", "rcdp202",
    "rdp103", "cdp104", "cdp102", "rdp203", "rcdp104",
]
CWD = "ph-showoa" if os.path.exists("ph-showoa") else "."
DATASET_DIR = "/kaggle/input/datasets/keith1101/ph-showoa/Wang_Chen"
OUTPUT_CSV = "/kaggle/working/summary_native_cuda_full_gpu.csv"
LOG_DIR = "/kaggle/working/logs_native_cuda"
RUNS = 30
MAX_ITER = 1000
POP_SIZE = 36
INIT_MODE = "sa_rcrs_grasp"
WORKERS = 1
RESUME = True
MAX_TIMEOUT_S = None
TAIL_LINES = 20
os.makedirs(LOG_DIR, exist_ok=True)

def find_file(name):
    candidates = [
        os.path.join(CWD, "dataset", f"explicit_{name}.vrpsdptw"),
        os.path.join(DATASET_DIR, f"explicit_{name}.vrpsdptw"),
    ]
    for pattern in (f"/kaggle/input/**/explicit_{name}.vrpsdptw", f"**/explicit_{name}.vrpsdptw"):
        candidates.extend(glob.glob(pattern, recursive=True))
    return next((path for path in candidates if os.path.isfile(path)), None)

def parse_output(text):
    if "Traceback (most recent call last)" in text or "Full-GPU solver failed:" in text:
        return None
    runs = re.search(r"Total (\d+) runs, total consumed ([\d.]+) sec", text)
    nv = re.search(r"(?:Vehicle count|vehicle \(route\) number):\s*(\d+)", text)
    cost = re.search(r"Total cost:\s*([\d.]+)", text)
    distance = re.search(r"Total distance:\s*([\d.]+)", text)
    if not (runs and nv and cost) or int(runs.group(1)) != RUNS:
        return None
    total_runs = int(runs.group(1))
    total_cost = float(cost.group(1))
    total_distance = float(distance.group(1)) if distance else total_cost - 2000.0 * int(nv.group(1))
    return {
        "best_NV": int(nv.group(1)),
        "best_TD": f"{total_distance:.4f}",
        "total_cost": f"{total_cost:.4f}",
        "avg_time_s": f"{float(runs.group(2)) / total_runs:.2f}",
        "total_runs": total_runs,
        "Status": "Success",
    }

def save_summary(rows):
    columns = ["Dataset", "best_NV", "best_TD", "total_cost", "avg_time_s", "wall_time", "total_runs", "Status", "Error"]
    frame = pd.DataFrame(rows)
    for column in columns:
        if column not in frame:
            frame[column] = "N/A"
    frame = frame[columns]
    frame.columns = ["Dataset", "Best NV", "Best TD", "Total Cost", "Avg/Run", "Wall Time", "Runs", "Status", "Error"]
    frame.to_csv(OUTPUT_CSV, index=False)

results = []
completed_names = set()
if RESUME and os.path.exists(OUTPUT_CSV):
    previous = pd.read_csv(OUTPUT_CSV)
    for _, row in previous.iterrows():
        if str(row.get("Status", "")).strip().lower() != "success":
            continue
        name = str(row.get("Dataset", "")).strip()
        results.append({"Dataset": name, "best_NV": row.get("Best NV", "N/A"), "best_TD": row.get("Best TD", "N/A"), "total_cost": row.get("Total Cost", "N/A"), "avg_time_s": row.get("Avg/Run", "N/A"), "wall_time": row.get("Wall Time", "N/A"), "total_runs": row.get("Runs", "N/A"), "Status": "Success", "Error": ""})
        completed_names.add(name)

for name in DATASETS:
    if name in completed_names:
        print(f"[RESUME] skip successful dataset: {name}")
        continue
    problem = find_file(name)
    if problem is None:
        results.append({"Dataset": name, "Status": "File Not Found", "Error": "dataset file not found"})
        save_summary(results)
        continue

    command = [sys.executable, "-u", "-m", "src_python_gpu_SA_RCRS_GRASP.main", "--problem", problem, "--compute_backend", "cuda", "--init", INIT_MODE, "--paper_flags", "--architecture", "full_gpu", "--objective", "lexicographic", "--grasp_alpha_lo", "0.10", "--grasp_alpha_hi", "0.40", "--sa_iterations", "25", "--runs", str(RUNS), "--max_iter", str(MAX_ITER), "--pop_size", str(POP_SIZE), "--workers", str(WORKERS)]
    print(f"\n[RUN] {name}: native CUDA full_gpu | target runs={RUNS}")
    run_started = T.time()
    last_report = run_started
    lines = []
    tail = deque(maxlen=TAIL_LINES)
    timed_out = False
    process = subprocess.Popen(command, cwd=CWD, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            lines.append(line)
            clean = line.rstrip()
            tail.append(clean)
            now = T.time()
            run_marker = re.search(r"Run (\d+) finishes", clean)
            total_marker = re.search(r"Total (\d+) runs, total consumed", clean)
            if run_marker:
                print(f"  [{now - run_started:.1f}s] completed run {run_marker.group(1)}/{RUNS}", flush=True)
            elif total_marker:
                print(f"  [{now - run_started:.1f}s] {clean}", flush=True)
            elif now - last_report >= 20:
                print(f"  [{now - run_started:.0f}s] running... last: {clean[:100]}", flush=True)
                last_report = now
            if MAX_TIMEOUT_S and now - run_started > MAX_TIMEOUT_S:
                timed_out = True
                process.kill()
                break
        process.wait()
    except Exception:
        process.kill()
        process.wait()
        raise

    wall = T.time() - run_started
    output = "".join(lines)
    with open(os.path.join(LOG_DIR, f"{name}.log"), "w", encoding="utf-8") as log:
        log.write(output)
    return_code = process.returncode
    parsed = parse_output(output) if return_code == 0 and not timed_out else None
    if parsed is None:
        error_lines = [line.strip() for line in lines if "failed:" in line.lower() or "error" in line.lower() or "traceback" in line.lower()]
        row = {"Dataset": name, "Status": "Timeout" if timed_out else "Failed", "Error": " | ".join(error_lines[-3:]) or f"native process exit code {return_code}", "wall_time": f"{wall:.1f}s"}
        print(f"[FAILED] {name}: {row['Error']}")
    else:
        row = parsed | {"Dataset": name, "wall_time": f"{wall:.1f}s", "Error": ""}
        completed_names.add(name)
        print(f"[OK] {name}: completed {parsed['total_runs']}/{RUNS} runs | NV={row['best_NV']} TD={row['best_TD']}")
    results.append(row)
    save_summary(results)

print(f"Finished. Results saved to {OUTPUT_CSV}")

## Cell 5 – Bảng tổng hợp kết quả & xuất CSV


In [ ]:
import os
import pandas as pd

OUTPUT_CSV = "/kaggle/working/summary_native_cuda_full_gpu.csv"

if os.path.exists(OUTPUT_CSV):
    df = pd.read_csv(OUTPUT_CSV)
    print("BẢNG TỔNG HỢP KẾT QUẢ BENCHMARK (NATIVE CUDA FULL-GPU)\n")
    print(df.to_string(index=False))
    print(f"\n[OK] Đã lưu bảng kết quả tại: {OUTPUT_CSV}")
else:
    print(f"Chưa tìm thấy file kết quả {OUTPUT_CSV}.")